# 🏆 CareerLens AI — Phase 5: Grounded Career Knowledge Base & Deterministic Fit Engine

**Objective**:
- Construct a grounded Career Knowledge Base across 24 career domains.
- Implement the Multi-Signal Deterministic Career Fit formula:
  $$\text{Career Fit} = 0.35 \times \text{Skill} + 0.20 \times \text{Project} + 0.20 \times \text{Semantic} + 0.10 \times \text{Education} + 0.10 \times \text{Experience} + 0.05 \times \text{Classification}$$
- Enforce the Anti-Hallucination Relevance Gate.
- Run case study evaluations on real candidate profiles.


In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
from lib.career.taxonomy import CAREER_TAXONOMY
from lib.career.engine import calculate_career_fit

print("Career fit engine modules loaded.")

Career fit engine modules loaded.


## 1. Structured Career Knowledge Base Inspection

In [2]:
print(f"Total Career Profiles in Knowledge Base: {len(CAREER_TAXONOMY)}")
for career_name, criteria in list(CAREER_TAXONOMY.items())[:4]:
    print(f"=== Career: {career_name} [{criteria['domain']}] ===")
    print(f"  Required Skills: {criteria['skills'][:6]}...")
    print(f"  Project Keywords: {criteria['project_keywords'][:5]}...")
    print(f"  Education Keywords: {criteria['education_keywords']}")
    print(f"  Experience Keywords: {criteria['experience_keywords']}\n")

Total Career Profiles in Knowledge Base: 13
=== Career: Software Engineer [INFORMATION-TECHNOLOGY] ===
  Required Skills: ['Python', 'Java', 'C++', 'JavaScript', 'TypeScript', 'SQL']...
  Project Keywords: ['api', 'web application', 'backend', 'full stack', 'microservices']...
  Education Keywords: ['computer science', 'software engineering', 'computer engineering', 'information technology', 'informatics']
  Experience Keywords: ['developer', 'software engineer', 'programmer', 'backend engineer', 'full stack engineer']

=== Career: Machine Learning Engineer [INFORMATION-TECHNOLOGY] ===
  Required Skills: ['Python', 'Machine Learning', 'Deep Learning', 'PyTorch', 'TensorFlow', 'Scikit-Learn']...
  Project Keywords: ['classification', 'regression', 'neural network', 'deep learning', 'nlp']...
  Education Keywords: ['computer science', 'artificial intelligence', 'data science', 'machine learning', 'computational engineering']
  Experience Keywords: ['machine learning engineer', 'ai engine

## 2. Deterministic 6-Signal Scoring Engine Demonstration

In [3]:
sample_candidate_profile = {
    "identity": {"name": "Alex Mercer", "email": "alex.mercer@ai.org"},
    "skills": [
        {"name": "Python", "normalized_name": "Python", "category": "Programming"},
        {"name": "PyTorch", "normalized_name": "PyTorch", "category": "ML/AI"},
        {"name": "Machine Learning", "normalized_name": "Machine Learning", "category": "ML/AI"},
        {"name": "Deep Learning", "normalized_name": "Deep Learning", "category": "ML/AI"},
        {"name": "SQL", "normalized_name": "SQL", "category": "Data"},
        {"name": "Docker", "normalized_name": "Docker", "category": "Cloud"}
    ],
    "education": [
        {"degree": "Master of Science in Computer Science", "field": "Artificial Intelligence", "institution": "Stanford University"}
    ],
    "experience": [
        {"job_title": "Machine Learning Engineer", "company": "DeepMind Lab"}
    ],
    "projects": [
        {"name": "Vision Transformer Model Training", "description": "Trained deep learning neural network for image classification using PyTorch."}
    ],
    "career_signal": {
        "dataset_category": "INFORMATION-TECHNOLOGY",
        "confidence": 0.94
    },
    "raw_text_snippet": "Alex Mercer Master of Science in Computer Science Artificial Intelligence Stanford Machine Learning Engineer DeepMind Lab Python PyTorch Deep Learning SQL Docker trained deep learning neural network for image classification."
}

fit_result = calculate_career_fit(sample_candidate_profile)

print("=== CANDIDATE EVALUATION RESULT ===")
print("Primary Classification:", fit_result['classification'])
print("\nTop 3 Career Recommendations (Deterministic Scoring):")
for idx, match in enumerate(fit_result['career_fit'][:3], 1):
    print(f"#{idx} {match['career']} — Total Fit: {match['total_fit']}%")
    print(f"   Breakdown: {match['breakdown']}")
    print(f"   Matched Skills: {match['evidence']['matched_skills']}")
    print(f"   Missing Skills: {match['evidence']['missing_skills']}")
    print(f"   Matched Projects: {match['evidence']['matched_projects']}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== CANDIDATE EVALUATION RESULT ===
Primary Classification: {'category': 'INFORMATION-TECHNOLOGY', 'confidence': 0.94}

Top 3 Career Recommendations (Deterministic Scoring):
#1 Machine Learning Engineer — Total Fit: 62.0%
   Breakdown: {'skill_match': 60.0, 'project_match': 50.0, 'semantic_match': 31.5, 'education_match': 100.0, 'experience_match': 100.0, 'classification_signal': 94.0}
   Matched Skills: ['Python', 'Machine Learning', 'Deep Learning', 'PyTorch', 'Docker', 'SQL']
   Missing Skills: ['TensorFlow', 'Scikit-Learn', 'NumPy', 'Pandas']
   Matched Projects: ['classification', 'neural network', 'deep learning', 'model training']

#2 Data Scientist — Total Fit: 32.7%
   Breakdown: {'skill_match': 33.3, 'project_match': 0.0, 'semantic_match': 31.5, 'education_match': 100.0, 'experience_match': 0.0, 'classification_signal': 94.0}
   Matched Skills: ['Python', 'SQL', 'Machine Learning']
   Missing Skills: ['Pandas', 'NumPy', 'Scikit-Learn', 'Matplotlib', 'Seaborn', 'Statistical Mo

## 3. Anti-Hallucination Relevance Gate Verification
Let's verify that candidate Alex (AI/ML Engineer) is NOT recommended unrelated careers like 'Human Resources Specialist' or 'Accountant', even if general words overlap.


In [4]:
for c in fit_result['career_fit']:
    if c['career'] in ['Human Resources Specialist', 'Accountant', 'Management Consultant']:
        print(f"Career: {c['career']}")
        print(f"  Total Fit: {c['total_fit']}% | Skill Match: {c['breakdown']['skill_match']}% | Project Match: {c['breakdown']['project_match']}%")
        assert c['total_fit'] < 25.0, f"Error: Irrelevant career {c['career']} scored too high!"

print("Relevance Gate verification passed: Unrelated careers successfully penalized.")

Relevance Gate verification passed: Unrelated careers successfully penalized.
